In [1]:
!pip install transformers datasets seqeval accelerate evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=822383715d9d0442ef14cb699d764b730c1da0f34a66f1f93e976b5ed6909f83
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


# **Data Preprocessing**

In [2]:
import pandas as pd
import re
from datasets import Dataset
from transformers import AutoTokenizer

# -----------------------
# Parameters
# -----------------------
# CamelBERT-Mix is ideal for health forums and clinical notes
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-mix"
MAX_LENGTH = 128

# Load your dataset
df = pd.read_csv('Full_NER_Dataset.csv')

# -----------------------
# 1️⃣ Improved Arabic Normalization
# -----------------------
def normalize_arabic(text):
    if not isinstance(text, str): return ""
    text = re.sub("[إأآ]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    # Remove Arabic diacritics (Tashkeel)
    text = re.sub(r"[\u064B-\u0652]", "", text)
    return text.strip()

df = df[df['Token'].notna()].copy()
df['Token'] = df['Token'].apply(normalize_arabic)
df = df[df['Token'] != ""].copy()

# -----------------------
# 2️⃣ Dynamic Label Mapping
# -----------------------
# This captures all categories: DISEASE, SYMPTOM, MEDICATION, PROCEDURE, BODY_PART
label_list = sorted(df['Entity tag'].unique().tolist())
if 'O' in label_list:
    label_list.remove('O')
    label_list = ['O'] + label_list  # Ensure 'O' is at index 0

label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

print(f"Detected Categories: {label_list}")

# -----------------------
# 3️⃣ Split into Sentences (Grouping Tokens)
# -----------------------
sentences = []
sentence_tokens = []
sentence_labels = []

# Clinical notes often use commas or periods as boundaries
punctuation = {'.', '!', '؟', '،', ','}

for token, label in zip(df['Token'], df['Entity tag']):
    sentence_tokens.append(token)
    sentence_labels.append(label)
    if token in punctuation:
        sentences.append({
            'tokens': sentence_tokens,
            'ner_tags': [label_to_id[l] for l in sentence_labels]
        })
        sentence_tokens = []
        sentence_labels = []

# Catch any remaining tokens
if sentence_tokens:
    sentences.append({
        'tokens': sentence_tokens,
        'ner_tags': [label_to_id[l] for l in sentence_labels]
    })

# -----------------------
# 4️⃣ Create & Split Dataset
# -----------------------
full_ds = Dataset.from_list(sentences)
ds_split = full_ds.train_test_split(test_size=0.2, seed=42)
train_val = ds_split['train'].train_test_split(test_size=0.1, seed=42)

train_ds = train_val['train']
val_ds = train_val['test']
test_ds = ds_split['test']

# -----------------------
# 5️⃣ Tokenize & Align (Crucial for Transformers)
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens (like [CLS]) get -100
            if word_idx is None:
                label_ids.append(-100)
            # Label only the first sub-token of a word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Following sub-tokens get -100 to ignore them in loss calculation
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Run Tokenization
train_tokenized = train_ds.map(tokenize_and_align_labels, batched=True)
val_tokenized = val_ds.map(tokenize_and_align_labels, batched=True)
test_tokenized = test_ds.map(tokenize_and_align_labels, batched=True)

print(f"✅ Preprocessing complete.")
print(f"Final Count -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Detected Categories: ['O', 'B-BODY_PART', 'B-DISEASE', 'B-MEDICATION', 'B-PROCEDURE', 'B-SYMPTOM', 'I-BODY_PART', 'I-DISEASE', 'I-MEDICATION', 'I-PROCEDURE', 'I-SYMPTOM']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/7329 [00:00<?, ? examples/s]

Map:   0%|          | 0/815 [00:00<?, ? examples/s]

Map:   0%|          | 0/2036 [00:00<?, ? examples/s]

✅ Preprocessing complete.
Final Count -> Train: 7329, Val: 815, Test: 2036


In [3]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# **Training Loop**

In [19]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
import evaluate
from seqeval.metrics import classification_report

# 1. Load Metric
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 2. Initialize Model
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

# 3. Training Arguments
training_args = TrainingArguments(
    output_dir="./camelbert-clinical-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.7,
    logging_dir='./logs',
    logging_steps=10,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# 4. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

# 5. EXECUTE TRAINING
print("🚀 Training started...")
trainer.train()
trainer.save_model("./best_arabic_clinical_ner_model")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-567949461.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Training started...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.167900,0.118620,0.798611,0.777778,0.788057,0.960697
2,0.065000,0.117046,0.787512,0.816425,0.801708,0.961363
3,0.055900,0.112454,0.791353,0.813527,0.802287,0.962918
4,0.048300,0.119976,0.811339,0.801932,0.806608,0.963473
5,0.016300,0.120117,0.821606,0.800966,0.811155,0.964583


# **Overall Test Evaluation**

In [20]:
# Level 1: The "Scoreboard" - Quick summary of performance on unseen data
print("--- 1. Final Results on Test Dataset ---")
test_results = trainer.evaluate(test_tokenized)

print(f"Overall Precision: {test_results['eval_precision']:.4f}")
print(f"Overall Recall:    {test_results['eval_recall']:.4f}")
print(f"Overall F1-Score:  {test_results['eval_f1']:.4f}")

--- 1. Final Results on Test Dataset ---


Overall Precision: 0.8518
Overall Recall:    0.8361
Overall F1-Score:  0.8439


# **Category-Wise Evaluation**

In [21]:
from seqeval.metrics import classification_report
import numpy as np

def get_detailed_report(dataset, name="Final Test Set"):
    # 1. Get predictions from the model
    output = trainer.predict(dataset)
    predictions = np.argmax(output.predictions, axis=2)
    labels = output.label_ids

    # 2. Clean up predictions: Remove -100 (special tokens) and map IDs to Labels
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # 3. Print the clinical category report
    print(f"\n--- 2. Clinical Category Report ({name}) ---")
    print(classification_report(true_labels, true_predictions))

# Run it
get_detailed_report(test_tokenized)


--- 2. Clinical Category Report (Final Test Set) ---
              precision    recall  f1-score   support

   BODY_PART       0.88      0.84      0.86       530
     DISEASE       0.90      0.92      0.91       322
  MEDICATION       0.84      0.82      0.83       361
   PROCEDURE       0.88      0.81      0.84       403
     SYMPTOM       0.81      0.83      0.82       763

   micro avg       0.85      0.84      0.84      2379
   macro avg       0.86      0.84      0.85      2379
weighted avg       0.85      0.84      0.84      2379



# **Linguistic Evaluation (MSA vs. Dialect)**

In [27]:
import torch

def predict_tokens(tokens_list):
    # Tokenize the input tokens
    tokenized_inputs_batch = tokenizer(
        tokens_list,
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    # Get word_ids BEFORE moving to device, as BatchEncoding object is lost after conversion to dict
    word_ids = tokenized_inputs_batch.word_ids(batch_index=0)

    # Move input tensors to the same device as the model
    device = model.device
    tokenized_inputs_on_device = {k: v.to(device) for k, v in tokenized_inputs_batch.items()}

    # Make predictions
    with torch.no_grad():
        outputs = model(**tokenized_inputs_on_device)
    predictions = outputs.logits.argmax(dim=-1).squeeze().tolist()

    # Align predictions with original tokens, ignoring special tokens and subwords
    aligned_predictions = []
    previous_word_idx = None
    for i, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx != previous_word_idx:
            aligned_predictions.append(id_to_label[predictions[i]])
        previous_word_idx = word_idx
    return aligned_predictions

def evaluate_subset(tokens_list, true_labels_list=None):
    preds = predict_tokens(tokens_list)
    print("\nSubset Inference:")
    for t, l in zip(tokens_list, preds):
        print(f"{t}\t{l}")

    if true_labels_list is not None:
        # The metric expects a list of lists for predictions and references
        results = metric.compute(
            predictions=[preds],
            references=[true_labels_list]
        )
        print("\nSubset Metrics:")
        print(f"Precision: {results['overall_precision']:.4f}")
        print(f"Recall:    {results['overall_recall']:.4f}")
        print(f"F1-score:  {results['overall_f1']:.4f}")

# -------------------------------
# 15. MSA & Dialectal Subset Testing
# -------------------------------
test_tokens_msa = [
    "يشعر", "المريض", "بـ", "حمى", "و", "تناول", "بانادول", "."
]

true_labels_msa = [
    "O", "O", "O", "B-SYMPTOM", "O", "O", "B-MEDICATION", "O"
]


# Dialectal Sentence: "My head hurts since yesterday and my eye is red, I took drops for the allergy."
test_tokens_dialect = [
    "رأسي", "بيوجعني", "من", "امبارح", "وعيني", "حمرا", "،",
    "خدت", "قطرة", "عشان", "الحساسية", "."
]

true_labels_dialect = [
    "B-BODY_PART", "O", "O", "O", "B-BODY_PART", "B-SYMPTOM", "O",
    "O", "B-MEDICATION", "O", "B-DISEASE", "O"
]

# Execute updated evaluations
evaluate_subset(test_tokens_msa, true_labels_msa)
evaluate_subset(test_tokens_dialect, true_labels_dialect)



Subset Inference:
يشعر	O
المريض	O
بـ	O
حمى	B-SYMPTOM
و	O
تناول	O
بانادول	B-MEDICATION
.	O

Subset Metrics:
Precision: 1.0000
Recall:    1.0000
F1-score:  1.0000

Subset Inference:
رأسي	O
بيوجعني	O
من	O
امبارح	O
وعيني	O
حمرا	O
،	O
خدت	O
قطرة	B-MEDICATION
عشان	O
الحساسية	B-SYMPTOM
.	O

Subset Metrics:
Precision: 0.5000
Recall:    0.2000
F1-score:  0.2857
